# 11. 집계구 공백(축4) + MCLP(축6)

## 이 노트북이 하는 일
집계구 위험도(노트북 10, 실측·2026-06)를 기준으로 **AED 공백 집계구**를 찾고(축4), 공백을 덮는 **신규 AED**를 MCLP로 고른다(축6). 100m 격자(추정) 라인을 **집계구(실측)** 라인으로 교체.

## 왜 이렇게 설계했나 (설계 이유)
- **왜 집계구 기준인가:** 위험도가 실측 집계구(78개)로 격상됐으므로, 공백·배치도 같은 단위로 맞춰야 일관된다.
- **왜 야간 커버인가:** 심정지 야간 대응이 관건 → 야간 접근 가능 AED만으로 커버 판정.
- **왜 R=150m인가:** 집계구 중앙 크기 ~157m. 집계구 중심에서 150m내 야간 AED면 그 집계구를 커버로 간주(골든타임 도보 근사).
- **왜 greedy MCLP인가:** 한정 예산으로 최대 위험 커버. 정확해(ILP) 근사, 의존성 없이 빠름.

## 데이터 출처
- 집계구 위험도: 노트북 10 (oa_risk). AED: 축4(15000652). 표고: 06 그래프.

In [ ]:
import os, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import geopandas as gpd, osmnx as ox
from scipy.spatial import cKDTree
import folium, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams["font.family"]="Malgun Gothic"; plt.rcParams["axes.unicode_minus"]=False
CRS_WGS, CRS_M = 4326, 5186
R = 150          # 커버 반경(m) — 집계구 중앙 크기 근사
N_NEW = 15       # 신규 AED 최대 개수
os.makedirs("outputs", exist_ok=True)

## 1. 집계구 위험도 + AED 로드

In [ ]:
oa = gpd.read_parquet("outputs/oa_risk.parquet")                                      # 노트북10 집계구 위험도(2026-06)
print("집계구:", len(oa), "| risk_norm:", round(oa["risk_norm"].min(),2), "~", round(oa["risk_norm"].max(),2))

aed = pd.read_csv("outputs/aed_donggu.csv").dropna(subset=["wgs84Lat","wgs84Lon"]).copy()  # AED 106
aed_g = gpd.GeoDataFrame(aed, geometry=gpd.points_from_xy(aed["wgs84Lon"], aed["wgs84Lat"]),
                         crs=CRS_WGS).to_crs(CRS_M)
def to_hhmm(v):
    try: return int(float(v))
    except: return None
def is_night(r):
    e=to_hhmm(r.get("monEndTme")); s=to_hhmm(r.get("monSttTme"))
    if e is None: return False
    if s==0 and e in (0,2400,2359): return True
    return e>=2200
aed_g["night"]=aed_g.apply(is_night, axis=1)
print("AED:", len(aed_g), "| 야간:", int(aed_g["night"].sum()))

## 2. 집계구별 최근접 야간 AED 거리 + 커버 판정

In [ ]:
cen = np.c_[oa["cx"].values, oa["cy"].values]                                         # 집계구 중심
def nearest(mask):
    pts=np.c_[aed_g.loc[mask,"geometry"].x.values, aed_g.loc[mask,"geometry"].y.values]
    if len(pts)==0: return np.full(len(oa), np.inf)
    return cKDTree(pts).query(cen, k=1)[0]                                            # 중심→최근접 AED 거리
oa["aed_dist_all"]  = nearest(aed_g.index)
oa["aed_dist_night"]= nearest(aed_g["night"].values)
oa["covered_night"] = oa["aed_dist_night"] <= R                                       # 야간 커버 여부
print("야간 커버 집계구: %d / %d (R=%dm)" % (oa["covered_night"].sum(), len(oa), R))

## 3. 공백 집계구 (고위험 & 야간 미커버) + 표고

In [ ]:
thr = oa["risk_norm"].quantile(0.70)                                                  # 상위 30%
oa["high_risk"]=oa["risk_norm"]>=thr
oa["gap"]=oa["high_risk"] & (~oa["covered_night"])
print(f"고위험 집계구(상위30%): {int(oa['high_risk'].sum())} | 그중 야간 공백: {int(oa['gap'].sum())}")
print("\n공백 집계구 소속 동:"); print(oa.loc[oa['gap'],'dong'].value_counts().to_string())

# 표고 (그래프 노드 elev) — 저지대 편중 확인
def _sf(x):
    try: return float(x)
    except: return float("nan")
G=ox.load_graphml("outputs/graph_drive_conn.graphml", node_dtypes={"elev":_sf})
oa["elev"]=[G.nodes[n].get("elev",np.nan) for n in oa["entry_node"]]
aed_nodes=ox.distance.nearest_nodes(G, X=aed_g.geometry.x.values, Y=aed_g.geometry.y.values)
aed_g["elev"]=[G.nodes[n].get("elev",np.nan) for n in aed_nodes]
ae=pd.to_numeric(aed_g["elev"],errors="coerce").dropna()
he=pd.to_numeric(oa.loc[oa["high_risk"],"elev"],errors="coerce").dropna()
print("\nAED 표고 중앙 %.0fm | 고위험 집계구 표고 중앙 %.0fm" % (ae.median(), he.median()))
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(ae,bins=20,alpha=0.6,label=f"AED(n={len(ae)})",color="#2c7fb8")
ax.hist(he,bins=15,alpha=0.6,label=f"고위험 집계구(n={len(he)})",color="#d7191c")
ax.set_xlabel("표고(m)"); ax.set_ylabel("빈도"); ax.legend(); ax.set_title("AED vs 고위험 집계구 표고")
fig.tight_layout(); fig.savefig("outputs/oa_elev_hist.png",dpi=120); plt.show()

## 4. MCLP — 공백 덮는 신규 AED (집계구 기준)

In [ ]:
demand = (oa["risk_norm"]>0) & (~oa["covered_night"])                                 # 수요 = 위험>0 이면서 야간 미커버 집계구
d_idx = np.where(demand.values)[0]                                                    # 수요 집계구의 인덱스
d_xy = cen[d_idx]                                                                     # 수요 집계구 중심 좌표
d_w = oa["risk_norm"].values[d_idx]                                                   # 수요 가중치 = 위험도(더 위험할수록 커버 가치↑)
tree = cKDTree(d_xy)                                                                  # 수요 좌표로 KD트리(빠른 반경 탐색)
cover = tree.query_ball_point(cen, R)                                                 # 후보(모든 집계구 중심)별 반경 R내 덮는 수요 목록
covered=np.zeros(len(d_idx),bool)                                                     # 각 수요가 덮였는지 표시
selected=[]                                                                           # 선택된 신규 AED 위치(집계구 index)
hist=[]                                                                               # (개수, 누적 커버 위험가중) 진행 기록
for _ in range(N_NEW):                                                                # 최대 N_NEW개까지 반복
    best_g,best_c=-1,-1                                                               # 이번 단계 최고 이득 후보
    for ci,dl in enumerate(cover):                                                    # 모든 후보 검토
        if not dl: continue                                                          # 덮는 수요가 없으면 건너뜀
        g=d_w[[j for j in dl if not covered[j]]].sum()                               # 아직 안 덮인 수요의 위험가중 합 = 새 이득
        if g>best_g: best_g,best_c=g,ci                                              # 최대 이득 후보 갱신
    if best_c<0 or best_g<=0: break                                                  # 더 덮을 게 없으면 종료
    for j in cover[best_c]: covered[j]=True                                          # 선택 후보가 덮는 수요를 커버 처리
    selected.append(best_c)                                                          # 후보 확정
    hist.append((len(selected), d_w[covered].sum()))                                 # 진행 기록(누적 커버)
tot=d_w.sum()                                                                         # 전체 수요 위험가중 합
print(f"수요 집계구: {len(d_idx)} (위험가중 {tot:.1f}) | 신규 {len(selected)}개 선정")
for n,cw in hist:                                                                     # 개수별 커버율 출력(한계효용 확인)
    print(f"  {n:2d}개 → 위험가중 {cw/tot*100:.0f}% 커버")
sel=oa.iloc[selected]                                                                 # 선택된 집계구들
res=pd.DataFrame({"rank":range(1,len(sel)+1),"dong":sel["dong"].values,"TOT_OA_CD":sel["TOT_OA_CD"].values,  # 결과 표(순위=중요도)
                  "p75_2026":sel["p75_2026"].round(0).values,"risk_norm":sel["risk_norm"].round(3).values,
                  "lon":sel.to_crs(4326).geometry.centroid.x.values,"lat":sel.to_crs(4326).geometry.centroid.y.values})
res.to_csv("outputs/oa_mclp_new_aed.csv", index=False, encoding="utf-8-sig")          # 신규 AED 후보 저장
print("\n[신규 AED 후보]"); print(res[["rank","dong","p75_2026","risk_norm"]].to_string(index=False))

## 5. 저장 + 지도

In [ ]:
keep=["TOT_OA_CD","dong","risk_norm","high_risk","aed_dist_night","covered_night","gap","elev","geometry"]
oa[keep].to_parquet("outputs/oa_gap.parquet")
try: oa[keep].to_file("outputs/oa_gap.gpkg", driver="GPKG")
except Exception as e: print("gpkg 경고:", e)

gw=oa.to_crs(4326); aedw=aed_g.to_crs(4326)                                           # 지도용 위경도 변환
def rcol(v):                                                                          # 위험도 → 색 (빨강=고위험)
    t=float(v); r=int(255*min(t+0.1,1)); g=int(200*(1-t)); b=int(60*(1-t)); return f"#{r:02x}{g:02x}{b:02x}"
m=folium.Map(location=[35.122,129.045], zoom_start=14, tiles="cartodbpositron")       # 배경 지도
for _,r in gw.iterrows():                                                             # ① 위험도 집계구 색칠
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x,c=rcol(r["risk_norm"]):{"color":"#555","weight":0.4,"fillColor":c,"fillOpacity":0.5}).add_to(m)
for _,r in gw[gw["gap"]].iterrows():                                                  # ② 공백 집계구 강조(검은 테두리)
    folium.GeoJson(r["geometry"].__geo_interface__,
        style_function=lambda x:{"color":"#000","weight":2.5,"fill":False}).add_to(m)
for _,a in aedw.iterrows():                                                           # ③ 기존 AED(초록=야간, 회색=주간)
    folium.CircleMarker([a.geometry.y,a.geometry.x],radius=3,color="#1a9641" if a["night"] else "#999",
        fill=True,fillOpacity=0.9).add_to(m)
for _,r in res.iterrows():                                                            # ④ 신규 AED 제안(파랑 별)
    folium.Marker([r["lat"],r["lon"]],tooltip=f'신규#{int(r["rank"])} {r["dong"]}',
        icon=folium.Icon(color="blue",icon="star",prefix="fa")).add_to(m)
legend=('<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;'
        'border:1px solid #999;font-size:12px"><b>집계구 공백·배치</b><br>'
        '<span style="color:#d7191c">&#9644;</span> 고위험 &nbsp; <span style="color:#000">▢</span> 공백<br>'
        '<span style="color:#1a9641">●</span> 야간AED &nbsp; <span style="color:blue">★</span> 신규</div>')
m.get_root().html.add_child(folium.Element(legend))
m.save("outputs/oa_gap_mclp_map.html")
print("저장: outputs/oa_gap.* + oa_mclp_new_aed.csv + oa_gap_mclp_map.html + oa_elev_hist.png")
m